# 模組 4 · 模型訓練與比較
## 多模型、超參數調校、堆疊集成、PLS 變體、TabPFN、AOM

學會比較多個模型、用 Optuna 自動調參、建立集成，並認識專為光譜設計的 PLS 變體與 AOM 家族。

**對應官方範例**：`examples/user/04_models/`（U01–U07）

## 🚀 環境設定（請先執行下面這格）

本課程使用 **nirs4all 官方範例資料集（sample_data）**。下面這格會：
1. 從 GitHub 下載 nirs4all（內含 0.9 版函式庫與官方光譜資料集）
2. 安裝函式庫
3. 切換到 `examples/` 目錄（裡面有 `sample_data/`）

> 💡 **為什麼從 GitHub 安裝？** PyPI（`pip install nirs4all`）目前是舊版 0.4，沒有本課程使用的 `nirs4all.run()` 簡易 API。GitHub 上的版本（0.9+）才有，且需要 **Python ≥ 3.11**（Colab 預設 3.12，沒問題）。

第一次執行約需 1–2 分鐘，請耐心等候出現「✅ 完成」。

In [ ]:
# === Colab / Jupyter 環境設定（每本 notebook 開頭執行一次）===
import os, sys, subprocess

if not os.path.isdir('nirs4all'):
    print('⬇️  下載 nirs4all 與官方資料集（約 1–2 分鐘）…')
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/GBeurier/nirs4all.git'], check=True)
    print('📦 安裝 nirs4all …')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    './nirs4all'], check=True)

# 切換到 examples 目錄（內含 sample_data）
if os.path.basename(os.getcwd()) != 'examples' and os.path.isdir('nirs4all/examples'):
    os.chdir('nirs4all/examples')

import nirs4all
print('✅ 完成！nirs4all 版本：', nirs4all.__version__)
print('   工作目錄：', os.getcwd())
print('   可用資料集：', [d for d in os.listdir('sample_data') if os.path.isdir(os.path.join('sample_data', d))])

---
## U01 · 多模型比較  ★★☆☆☆

在 pipeline 放多個 `{"model": ...}`，nirs4all 自動訓練並排名。NIRS 常用：線性（PLS、Ridge、ElasticNet）、樹（RandomForest、GradientBoosting）。**PLS 最適合高度共線的光譜資料**。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from nirs4all.operators.transforms import StandardNormalVariate

pipeline = [
    StandardNormalVariate(), MinMaxScaler(), {"y_processing": MinMaxScaler()},
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {"model": PLSRegression(n_components=10),       "name": "PLS"},
    {"model": Ridge(alpha=1.0),                     "name": "Ridge"},
    {"model": RandomForestRegressor(random_state=42), "name": "RandomForest"},
    {"model": GradientBoostingRegressor(random_state=42), "name": "GradBoost"},
]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="MultiModel", verbose=1)
for i, p in enumerate(result.top(n=4, display_metrics=['rmse', 'r2']), 1):
    print(f"{i}. {p['model_name']:14s} RMSE {p['rmse']:.4f} | R² {p['r2']:.4f}")

> ✍️ **練習**：PLS 在此資料是否贏過樹模型？換成 `sample_data/regression_2` 結論是否相同？

---
## U02 · 超參數調校：Optuna  ★★★☆☆

用 `finetune_params` 內建 Optuna 自動最佳化。搜尋法：grid / random / tpe / binary / auto。參數型別：`('int',min,max)`、`('float_log',...)`、`[類別]`。PLS 成分數這種單峰整數用 `binary` 最快。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import StandardNormalVariate

pipeline = [
    StandardNormalVariate(), MinMaxScaler(), {"y_processing": MinMaxScaler()},
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {
        "model": PLSRegression(),
        "finetune_params": {
            "n_trials": 20,
            "sampler": "tpe",
            "metric": "rmse",
            "model_params": {
                "n_components": ('int', 1, 20),
                "scale": [True, False],
            },
        },
    },
]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="Tuning", verbose=1)
print("\n最佳 RMSE:", round(result.best_rmse, 4))

> ✍️ **練習**：把 `sampler` 改成 `"binary"` 調 `n_components`，比較達到相近 RMSE 所需的 trial 數。

---
## U03 · 堆疊集成：Stacking 與 Voting  ★★★☆☆

集成把多個模型組合以提升泛化。**Stacking** 用 meta-learner 學習組合；**Voting** 做平均。模型越多樣（線性 + 樹）增益越大，但犧牲可解釋性。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.ensemble import (StackingRegressor, VotingRegressor,
    RandomForestRegressor, GradientBoostingRegressor)
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from nirs4all.operators.transforms import StandardNormalVariate

stack = StackingRegressor(
    estimators=[('pls', PLSRegression(n_components=10)),
                ('rf',  RandomForestRegressor(random_state=42)),
                ('gbr', GradientBoostingRegressor(random_state=42))],
    final_estimator=Ridge(alpha=1.0), cv=3)
vote = VotingRegressor(
    estimators=[('pls', PLSRegression(n_components=10)),
                ('rf',  RandomForestRegressor(random_state=42))])

pipeline = [
    StandardNormalVariate(), MinMaxScaler(), {"y_processing": MinMaxScaler()},
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {"model": PLSRegression(n_components=10), "name": "PLS(單一)"},
    {"model": stack, "name": "Stacking"},
    {"model": vote,  "name": "Voting"},
]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="Ensemble", verbose=1)
for p in result.top(n=3, display_metrics=['rmse']):
    print(f"{p['model_name']:12s} RMSE {p['rmse']:.4f}")

> ✍️ **練習**：集成是否真的贏過最佳單一模型？評估多出的運算成本是否值得。

---
## U04 · PLS 變體  ★★★★☆

PLS 家族針對不同情境有專門變體：`IKPLS`（大資料快速）、`OPLS`（濾 Y-正交變異）、`SparsePLS`（變數選擇）、`KernelPLS`（非線性）、`PLSDA`（分類）。本課示範標準 PLS 與（若有安裝）SparsePLS。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import StandardNormalVariate

pipeline = [
    StandardNormalVariate(), MinMaxScaler(), {"y_processing": MinMaxScaler()},
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {"model": PLSRegression(n_components=5),  "name": "PLS-5"},
    {"model": PLSRegression(n_components=15), "name": "PLS-15"},
]

# 嘗試載入進階 PLS 變體（如未安裝會自動略過）
try:
    from nirs4all.operators.models import SparsePLS
    pipeline.append({"model": SparsePLS(n_components=10), "name": "SparsePLS"})
except Exception as e:
    print("（SparsePLS 不可用，略過）", e)

result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="PLSVariants", verbose=1)
for p in result.top(n=5, display_metrics=['rmse']):
    print(f"{p['model_name']:12s} RMSE {p['rmse']:.4f}")

> ✍️ **練習**：`!pip install ikpls` 後試 `from nirs4all.operators.models import IKPLS`，比較速度與精度。

---
## U05 · 進階微調：複雜最佳化  ★★★★☆

進階 HPO：多階段（先廣探索再精利用）、自訂指標與方向、**force_params**（把已知好設定當種子加速收斂）、dict 格式參數（支援 step/log）。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import StandardNormalVariate

pipeline = [
    StandardNormalVariate(), MinMaxScaler(), {"y_processing": MinMaxScaler()},
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {
        "model": PLSRegression(),
        "finetune_params": {
            "n_trials": 25, "metric": "rmse", "direction": "minimize",
            "model_params": {
                "n_components": {'type': 'int', 'min': 1, 'max': 25},
            },
            "force_params": [{"n_components": 12}],   # 種子已知好設定
        },
    },
]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="AdvTuning", verbose=1)
print("最佳 RMSE:", round(result.best_rmse, 4))

> ✍️ **練習**：把 U02 找到的最佳 `n_components` 放進 `force_params`，看是否更快收斂。

---
## U06 · TabPFN：免調參表格基礎模型  ★★☆☆☆

`TabPFNNIRSRegressor` 採固定配方（SG 一階導數 + OSC + 置中 + TabPFN），免逐資料集 HPO、開箱即用。在 57 個 NIRS 資料集上中位 RMSE 僅比逐資料集 HPO 高約 2.5%。需 `pip install tabpfn`，大樣本建議 GPU。

In [ ]:
# TabPFN 為可選相依，第一次需安裝（Colab 有 GPU 時更快）
try:
    from nirs4all.operators.models import TabPFNNIRSRegressor
    import nirs4all
    from sklearn.model_selection import ShuffleSplit
    pipeline = [ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
                {"model": TabPFNNIRSRegressor(n_estimators=16), "name": "TabPFN"}]
    result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                          name="TabPFN", verbose=1)
    print("TabPFN RMSE:", round(result.best_rmse, 4))
except Exception as e:
    print("TabPFN 尚未安裝。請先執行：!pip install tabpfn")
    print("錯誤訊息：", e)

> ✍️ **練習**：安裝 tabpfn 後，比較 TabPFN 與最佳 PLS 的 RMSE 與訓練時間。

---
## U07 · AOM 全家桶：自動運算子選擇  ★★★★★

**AOM**（Adaptive Operator-Mixture）家族會自動從運算子庫選出最佳前處理組合。成員：`AOMPLSRegressor`、`AOMRidgeRegressor`、`AOMRidgeAutoSelector`、`AOMRidgeBlender`（最強實證配方）、`FastAOMPLSRidge`（加速版）。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import StandardNormalVariate

pipeline = [
    StandardNormalVariate(), MinMaxScaler(), {"y_processing": MinMaxScaler()},
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {"model": PLSRegression(n_components=5), "name": "PLS-基準"},
]
# 嘗試加入 AOM-PLS（自動運算子選擇）
try:
    from nirs4all.operators.models import AOMPLSRegressor
    pipeline.append({
        "model": AOMPLSRegressor(n_components="auto", max_components=5,
                 operator_bank="compact", criterion="cv", cv=4, random_state=42),
        "name": "AOM-PLS"})
except Exception as e:
    print("（AOM 模型不可用，略過）", e)

result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="AOM", verbose=1)
for p in result.top(n=3, display_metrics=['rmse']):
    print(f"{p['model_name']:12s} RMSE {p['rmse']:.4f}")

> ✍️ **練習**：AOM-PLS 自動選的前處理，能否打敗你在模組 3 手動找到的最佳前處理鏈？